## 🔷 Phase 0: Setup and Environment

### 🎯 Objective

Set up the development environment by installing dependencies, importing required libraries, and ensuring reproducibility through fixed random seeds.

---

### ⚙️ What Happens in This Phase

* Install all required Python packages (e.g., PyTorch, NumPy, Librosa)
* Import essential libraries for:

  * Audio processing
  * Data handling
  * Model building and training
* Define a **global seed function** to control randomness across:

  * Python
  * NumPy
  * PyTorch (CPU & GPU)

---

### ❓ Why This Phase Is Important

* Ensures **reproducibility** of results across runs
* Provides a **consistent experimental setup**
* Simplifies debugging by eliminating randomness-related variations
* Establishes a **clean and organized starting point** for the pipeline

---

### 💡 Key Insight

Without fixing random seeds, model performance can vary between runs, making it difficult to compare experiments reliably.


In [ ]:
# Phase 0 — Install required libraries
!pip install -q transformers librosa torchmetrics wandb

In [ ]:
# Phase 0 — Imports

import os
import random
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import librosa
import wandb
from kaggle_secrets import UserSecretsClient
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import (
    ASTFeatureExtractor,
    ASTForAudioClassification,
    get_cosine_schedule_with_warmup,
)
from torchmetrics.classification import MulticlassF1Score

warnings.filterwarnings('ignore')
print('All imports done!')

In [ ]:
user_secrets = UserSecretsClient()
secret_value_1 = user_secrets.get_secret("WANDB_API_KEY")

# Expose it as a normal env var so existing code continues to work
os.environ["WANDB_API_KEY"] = secret_value_1

print("WANDB_API_KEY set from Kaggle secrets. Length:", len(secret_value_1))

In [ ]:
# Phase 0 — Reproducibility seed

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
print('Seed set!')

## 🔷 Phase 1: Configuration and Data Loading

### 🎯 Objective

Define all experiment configurations, initialize tracking (e.g., W&B), organize dataset paths, and prepare a stratified train/validation split.

---

### ⚙️ What Happens in This Phase

* Define **global hyperparameters**, including:

  * Audio duration
  * Batch size
  * Learning rates
  * Augmentation settings

* Initialize **experiment tracking configuration** (e.g., Weights & Biases)

* Set up:

  * Dataset directory paths
  * Genre label mappings

* Build a **song index**:

  * Organizes audio samples by genre
  * Validates that all required stems (e.g., multiple audio components) are present per track

* Load additional **noise samples** (e.g., from ESC-50 dataset) for augmentation

* Create an **85/15 stratified train/validation split**:

  * Preserves class distribution across splits

---

### ❓ Why This Phase Is Important

* Centralized configuration makes experiments **easy to tune and reproduce**
* Proper dataset structuring prevents **data leakage and missing samples**
* Stratified splitting ensures **balanced evaluation across all genres**
* External noise data improves **model robustness through augmentation**

---

### 💡 Key Insight

A well-designed configuration and data pipeline is critical for reliable experiments—poor data handling can invalidate even the best models.


In [ ]:
# Phase 1 — Hyperparameters and config

# DURATION    : seconds of audio fed per sample
# TRAIN_SIZE  : synthetic samples generated per epoch
# EPOCHS_P2   : full fine-tune epochs (Phase 2)
# LR_P2       : must be small to protect pretrained weights
# N_TTA       : random crops averaged at inference
# NOISE_PROB  : probability of adding ESC-50 noise

SAMPLE_RATE  = 16000
DURATION     = 20
MAX_LENGTH   = SAMPLE_RATE * DURATION

BATCH_SIZE   = 6
GRAD_ACCUM   = 2
NUM_WORKERS  = 2

EPOCHS_P1    = 3
EPOCHS_P2    = 5
LR_P1        = 3e-4
LR_P2        = 2e-5
LR_HEAD_MULT = 10
WEIGHT_DECAY = 0.01

TRAIN_SIZE   = 5000
NOISE_PROB   = 0.7
NOISE_LEVEL  = (0.03, 0.20)
TEMPO_PROB   = 0.4
TEMPO_RANGE  = (0.88, 1.12)
PATIENCE_P1  = 2
PATIENCE_P2  = 3

N_TTA        = 5

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

WANDB_PROJECT  = 'music-genre-classification-1'
WANDB_RUN_NAME = f'AST_dur{DURATION}_bs{BATCH_SIZE}_p2ep{EPOCHS_P2}'

print(f'Device      : {DEVICE}')
print(f'Duration    : {DURATION}s')
print(f'Train size  : {TRAIN_SIZE}')
print(f'Epochs P1   : {EPOCHS_P1} | P2: {EPOCHS_P2}')
print(f'LR P1       : {LR_P1}  | P2: {LR_P2}')
print(f'TTA crops   : {N_TTA}')
print(f'W&B project : {WANDB_PROJECT}')

In [ ]:
# Phase 1 — W&B initialisation

# Kaggle: set WANDB_API_KEY as a secret (Add-ons -> Secrets)
wandb.login(key=os.environ.get('WANDB_API_KEY', ''))

wandb.init(
    project = WANDB_PROJECT,
    name    = WANDB_RUN_NAME,
    config  = {
        'duration'      : DURATION,
        'sample_rate'   : SAMPLE_RATE,
        'batch_size'    : BATCH_SIZE,
        'grad_accum'    : GRAD_ACCUM,
        'epochs_p1'     : EPOCHS_P1,
        'epochs_p2'     : EPOCHS_P2,
        'lr_p1'         : LR_P1,
        'lr_p2'         : LR_P2,
        'lr_head_mult'  : LR_HEAD_MULT,
        'weight_decay'  : WEIGHT_DECAY,
        'train_size'    : TRAIN_SIZE,
        'noise_prob'    : NOISE_PROB,
        'tempo_prob'    : TEMPO_PROB,
        'n_tta'         : N_TTA,
        'model'         : 'MIT/ast-finetuned-audioset-10-10-0.4593',
    }
)

print(f'W&B run started : {wandb.run.name}')
print(f'Project         : {WANDB_PROJECT}')

In [ ]:
# Phase 1 — Paths and label mappings

BASE_PATH      = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
GENRES_PATH    = os.path.join(BASE_PATH, 'genres_stems')
ESC_PATH       = os.path.join(BASE_PATH, 'ESC-50-master', 'audio')
REQUIRED_STEMS = ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']

GENRES     = ['blues', 'classical', 'country', 'disco', 'hiphop',
              'jazz', 'metal', 'pop', 'reggae', 'rock']
label2id   = {g: i for i, g in enumerate(GENRES)}
id2label   = {i: g for g, i in label2id.items()}
NUM_LABELS = len(GENRES)

print('Paths set!')
print('label2id:', label2id)

In [ ]:
# Phase 1 — Build song index and collect ESC-50 noise files

def build_song_index():
    index = {}
    for genre in GENRES:
        genre_path = os.path.join(GENRES_PATH, genre)
        valid = []
        for folder in sorted(os.listdir(genre_path)):
            folder_path = os.path.join(genre_path, folder)
            if not os.path.isdir(folder_path):
                continue
            if all(os.path.exists(os.path.join(folder_path, s))
                   for s in REQUIRED_STEMS):
                valid.append(folder_path)
        index[genre] = valid
        print(f'  {genre}: {len(valid)} songs')
    return index

song_index = build_song_index()

noise_files = []
for root, _, files in os.walk(ESC_PATH):
    for f in files:
        if f.endswith('.wav'):
            noise_files.append(os.path.join(root, f))

print(f'\nNoise files: {len(noise_files)}')

wandb.log({
    'dataset/total_songs' : sum(len(v) for v in song_index.values()),
    'dataset/noise_files' : len(noise_files),
    'dataset/genres'      : len(GENRES),
})

In [ ]:
# Phase 1 — Train/validation split (per genre, 85/15)

train_index = {}
val_index   = {}

for genre in GENRES:
    songs = song_index[genre][:]
    random.shuffle(songs)
    split              = int(0.85 * len(songs))
    train_index[genre] = songs[:split]
    val_index[genre]   = songs[split:]
    print(f'  {genre} -> Train: {len(train_index[genre])}, Val: {len(val_index[genre])}')

print('\nTrain/Val split done!')

## 🔷 Phase 2: Audio Preprocessing and Feature Engineering

### 🎯 Objective

Design and implement reusable functions for audio loading, normalization, cropping, augmentation, and noise mixing.

---

### ⚙️ What Happens in This Phase

* Implement **audio loading utilities**:

  * Read waveform files
  * Convert to a consistent sample rate
  * Normalize amplitude

* Apply **cropping strategies**:

  * **Random cropping** for training (data diversity)
  * **Center cropping** for validation (consistency)

* Define **augmentation techniques**:

  * Random gain (volume variation)
  * Tempo/speed perturbation
  * Time-domain transformations

* Integrate **noise mixing**:

  * Add background noise samples (e.g., ESC-50 dataset)
  * Control noise level for realistic augmentation

* Package all steps into **reusable helper functions** for pipeline consistency

---

### ❓ Why This Phase Is Important

* Separates **audio processing logic** from model architecture
* Ensures **consistent and reusable preprocessing pipeline**
* Improves **model generalization** through augmentation
* Prevents overfitting by introducing **controlled variability**
* Maintains evaluation stability using deterministic validation preprocessing

---

### 💡 Key Insight

Using different preprocessing strategies for training (randomized) and validation (deterministic) helps the model generalize better while keeping evaluation reliable and comparable.


In [ ]:
# Phase 2 — Audio helper functions

def load_audio(path):
    audio, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    return audio.astype(np.float32)

def normalize(audio):
    return audio / (np.max(np.abs(audio)) + 1e-6)

def crop_random(audio):
    if len(audio) >= MAX_LENGTH:
        start = random.randint(0, len(audio) - MAX_LENGTH)
        return audio[start: start + MAX_LENGTH]
    return np.pad(audio, (0, MAX_LENGTH - len(audio)))

def crop_center(audio):
    if len(audio) >= MAX_LENGTH:
        start = (len(audio) - MAX_LENGTH) // 2
        return audio[start: start + MAX_LENGTH]
    return np.pad(audio, (0, MAX_LENGTH - len(audio)))

def random_gain(audio):
    return audio * random.uniform(0.7, 1.3)

def tempo_augment(audio):
    if random.random() < TEMPO_PROB:
        rate  = random.uniform(*TEMPO_RANGE)
        audio = librosa.effects.time_stretch(audio, rate=rate)
    return audio

def add_noise(audio):
    if random.random() < NOISE_PROB and len(noise_files) > 0:
        noise = load_audio(random.choice(noise_files))
        noise = crop_random(noise)
        level = random.uniform(*NOISE_LEVEL)
        audio = audio + level * noise
    return audio

print('Audio helpers ready!')

## 🔷 Phase 3: Model and Dataset Construction

### 🎯 Objective

Initialize the pretrained Audio Spectrogram Transformer (AST), define custom training and validation datasets, and set up DataLoaders for efficient batching.

---

### ⚙️ What Happens in This Phase

* Load **pretrained AST model**:

  * Replace the original classification head with a **10-class output layer**
  * Enable fine-tuning for the GTZAN task

* Define **custom Dataset classes**:

  * **TrainDataset**:

    * Generates synthetic mashups using multiple audio stems
    * Applies augmentation (cropping, noise, gain, etc.)
  * **ValDataset**:

    * Loads clean audio samples
    * Uses deterministic **center cropping** (no augmentation)

* Create **DataLoaders**:

  * Batch data efficiently
  * Shuffle training data for better generalization
  * Use parallel loading for improved performance

---

### ❓ Why This Phase Is Important

* Pretrained AST provides **rich audio feature representations**, improving performance on small datasets
* Custom Dataset classes encapsulate complex data logic, keeping training code **clean and modular**
* DataLoaders optimize **training speed and memory usage**
* Separate training and validation pipelines ensure **robust evaluation**

---

### 💡 Key Insight

Combining pretrained models with well-designed dataset pipelines significantly boosts performance while keeping the system scalable and maintainable.


In [ ]:
# Phase 3 — Load pretrained AST model

MODEL_NAME = 'MIT/ast-finetuned-audioset-10-10-0.4593'

feature_extractor = ASTFeatureExtractor.from_pretrained(MODEL_NAME)
print('Feature extractor loaded!')

model = ASTForAudioClassification.from_pretrained(
    MODEL_NAME,
    num_labels              = NUM_LABELS,
    id2label                = id2label,
    label2id                = label2id,
    ignore_mismatched_sizes = True,
)
model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f'AST loaded on {DEVICE}')
print(f'Total parameters: {total_params:,}')

wandb.log({'model/total_params': total_params})

In [ ]:
# Phase 3 — TrainDataset: on-the-fly synthetic mashups

class TrainDataset(Dataset):
    def __init__(self, song_index, feature_extractor, size=TRAIN_SIZE):
        self.song_index        = song_index
        self.feature_extractor = feature_extractor
        self.size              = size

    def __len__(self):
        return self.size

    def __getitem__(self, idx):
        while True:
            try:
                genre = random.choice(GENRES)
                songs = self.song_index[genre]
                if len(songs) == 0:
                    continue

                mixed = None
                for stem in REQUIRED_STEMS:
                    song_path = random.choice(songs)
                    audio     = load_audio(os.path.join(song_path, stem))
                    audio     = crop_random(audio)
                    audio     = tempo_augment(audio)
                    audio     = random_gain(audio)
                    mixed     = audio if mixed is None else mixed + audio

                mixed  = normalize(mixed)
                mixed  = add_noise(mixed)
                mixed  = normalize(mixed)

                inputs = self.feature_extractor(
                    mixed,
                    sampling_rate  = SAMPLE_RATE,
                    return_tensors = 'pt'
                )
                return {
                    'input_values': inputs['input_values'].squeeze(0),
                    'labels'      : torch.tensor(label2id[genre], dtype=torch.long)
                }
            except Exception:
                continue

print('TrainDataset defined!')

In [ ]:
# Phase 3 — ValDataset: clean center-cropped evaluation

class ValDataset(Dataset):
    def __init__(self, song_index, feature_extractor):
        self.feature_extractor = feature_extractor
        self.samples = []
        for genre in GENRES:
            for song_path in song_index[genre]:
                self.samples.append((genre, song_path))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        genre, song_path = self.samples[idx]
        mixed = None
        for stem in REQUIRED_STEMS:
            audio = load_audio(os.path.join(song_path, stem))
            audio = crop_center(audio)
            mixed = audio if mixed is None else mixed + audio

        mixed  = normalize(mixed)
        inputs = self.feature_extractor(
            mixed,
            sampling_rate  = SAMPLE_RATE,
            return_tensors = 'pt'
        )
        return {
            'input_values': inputs['input_values'].squeeze(0),
            'labels'      : torch.tensor(label2id[genre], dtype=torch.long)
        }

print('ValDataset defined!')

In [ ]:
# Phase 3 — DataLoaders

train_dataset = TrainDataset(train_index, feature_extractor, size=TRAIN_SIZE)
val_dataset   = ValDataset(val_index, feature_extractor)

train_loader = DataLoader(
    train_dataset,
    batch_size         = BATCH_SIZE,
    shuffle            = True,
    num_workers        = NUM_WORKERS,
    pin_memory         = True,
    persistent_workers = True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size         = BATCH_SIZE,
    shuffle            = False,
    num_workers        = NUM_WORKERS,
    pin_memory         = True,
    persistent_workers = True,
)

print(f'Train samples : {len(train_dataset)}')
print(f'Val samples   : {len(val_dataset)}')
print(f'Train batches : {len(train_loader)}')
print(f'Val batches   : {len(val_loader)}')

wandb.log({
    'dataset/train_samples' : len(train_dataset),
    'dataset/val_samples'   : len(val_dataset),
})

## 🔷 Phase 4: Training Pipeline

### 🎯 Objective

Define the loss function, evaluation metric, and implement structured training and validation loops.

---

### ⚙️ What Happens in This Phase

* Define **loss function**:

  * Cross-entropy loss with **label smoothing** to improve generalization

* Define **evaluation metric**:

  * **Macro F1-score** using `torchmetrics`
  * Ensures equal importance across all classes

* Implement **training loop (`train_one_epoch`)**:

  * Forward pass + loss computation
  * Backpropagation with **gradient accumulation**
  * Optimizer updates
  * Logging metrics to **Weights & Biases (W&B)**

* Implement **validation loop (`validate`)**:

  * Disable gradients for efficiency
  * Compute predictions on validation set
  * Calculate **Macro F1-score**

---

### ❓ Why This Phase Is Important

* Encapsulates training logic in a **clean and reusable structure**
* Label smoothing reduces **overconfidence and overfitting**
* Macro F1 aligns with **Kaggle-style evaluation**, especially for imbalanced classes
* Separate validation ensures **unbiased performance tracking**
* Logging enables **experiment monitoring and comparison**

---

### 💡 Key Insight

Optimizing for the right metric (Macro F1 instead of accuracy) ensures the model performs well across all classes—not just the dominant ones.


In [ ]:
# Phase 4 — Loss and metric

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
f1_metric = MulticlassF1Score(
    num_classes = NUM_LABELS,
    average     = 'macro'
).to(DEVICE)

print('Loss   : CrossEntropyLoss (label_smoothing=0.1)')
print('Metric : Macro F1')

In [ ]:
# Phase 4 — Train and validate functions

def train_one_epoch(model, loader, optimizer, scheduler, epoch, phase):
    model.train()
    f1_metric.reset()
    total_loss = 0
    optimizer.zero_grad()

    for step, batch in enumerate(tqdm(loader, desc='Training')):
        input_values = batch['input_values'].to(DEVICE)
        labels       = batch['labels'].to(DEVICE)

        outputs = model(input_values=input_values)
        loss    = criterion(outputs.logits, labels) / GRAD_ACCUM
        loss.backward()

        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

            wandb.log({
                f'{phase}/step_loss' : loss.item() * GRAD_ACCUM,
                f'{phase}/lr'        : scheduler.get_last_lr()[0],
            })

        total_loss += loss.item() * GRAD_ACCUM
        preds = torch.argmax(outputs.logits, dim=1)
        f1_metric.update(preds, labels)

    return total_loss / len(loader), f1_metric.compute().item()


def validate(model, loader):
    model.eval()
    f1_metric.reset()
    with torch.no_grad():
        for batch in tqdm(loader, desc='Validation'):
            input_values = batch['input_values'].to(DEVICE)
            labels       = batch['labels'].to(DEVICE)
            outputs      = model(input_values=input_values)
            preds        = torch.argmax(outputs.logits, dim=1)
            f1_metric.update(preds, labels)
    return f1_metric.compute().item()


print('train_one_epoch and validate ready!')

## 🔷 Phase 5: Two-Phase Fine-Tuning

### 🎯 Objective

Gradually adapt the pretrained Audio Spectrogram Transformer (AST) to the target task by first training the classification head and then fine-tuning the entire model.

---

### ⚙️ What Happens in This Phase

#### **Phase 1: Head Warm-Up**

* Freeze all pretrained backbone layers
* Train only the **final classification head**
* Use a relatively higher learning rate for faster adaptation

#### **Phase 2: Full Fine-Tuning**

* Unfreeze the entire model
* Apply **differential learning rates**:

  * Lower LR for pretrained backbone
  * Higher LR (≈10×) for the classification head
* Continue training with:

  * Learning rate scheduling
  * **Early stopping** to prevent overfitting

---

### ❓ Why This Phase Is Important

* Prevents disruption of **pretrained feature representations**
* Allows the model to first **specialize the classifier** before global tuning
* Differential learning rates ensure **stable and efficient optimization**
* Early stopping avoids **overfitting on small datasets like GTZAN**

---

### 💡 Key Insight

Directly fine-tuning all layers from the start can degrade pretrained knowledge—progressive unfreezing ensures a stable and effective transfer learning process.


In [ ]:
# Phase 5 — Phase 1: freeze base, train head only

for param in model.base_model.parameters():
    param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Phase 1: {trainable:,} trainable params (head only)')

optimizer_p1   = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr           = LR_P1,
    weight_decay = WEIGHT_DECAY
)
total_steps_p1 = (len(train_loader) // GRAD_ACCUM) * EPOCHS_P1
scheduler_p1   = get_cosine_schedule_with_warmup(
    optimizer_p1,
    num_warmup_steps   = total_steps_p1 // 10,
    num_training_steps = total_steps_p1
)

best_p1_f1   = 0
patience_cnt = 0

print(f'\n{"="*50}')
print(f'PHASE 1 — Classifier Warmup ({EPOCHS_P1} epochs)')
print(f'{"="*50}\n')

for epoch in range(EPOCHS_P1):
    train_loss, train_f1 = train_one_epoch(
        model, train_loader, optimizer_p1, scheduler_p1,
        epoch=epoch, phase='phase1'
    )
    val_f1 = validate(model, val_loader)

    wandb.log({
        'phase1/epoch'      : epoch + 1,
        'phase1/train_loss' : train_loss,
        'phase1/train_f1'   : train_f1,
        'phase1/val_f1'     : val_f1,
    })

    print(f'Epoch {epoch+1}/{EPOCHS_P1}')
    print(f'  Train Loss : {train_loss:.4f}')
    print(f'  Train F1   : {train_f1:.4f}')
    print(f'  Val F1     : {val_f1:.4f}')

    if val_f1 > best_p1_f1:
        best_p1_f1   = val_f1
        patience_cnt = 0
        torch.save(model.state_dict(), '/kaggle/working/best_model_phase1.pth')
        print(f'  Saved best Phase 1 model (val F1: {best_p1_f1:.4f})')
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE_P1:
            print('  Early stopping Phase 1.')
            break

wandb.run.summary['phase1_best_val_f1'] = best_p1_f1
print(f'\nPhase 1 done. Best val F1: {best_p1_f1:.4f}')

In [ ]:
# Phase 5 — Phase 2: unfreeze all, fine-tune with layer-wise LR

model.load_state_dict(
    torch.load('/kaggle/working/best_model_phase1.pth')
)

for param in model.parameters():
    param.requires_grad = True

total_params = sum(p.numel() for p in model.parameters())
print(f'All {total_params:,} params unfrozen!')

optimizer_p2 = torch.optim.AdamW([
    {'params': model.base_model.parameters(),  'lr': LR_P2},
    {'params': model.classifier.parameters(),  'lr': LR_P2 * LR_HEAD_MULT},
], weight_decay=WEIGHT_DECAY)

total_steps_p2 = (len(train_loader) // GRAD_ACCUM) * EPOCHS_P2
scheduler_p2   = get_cosine_schedule_with_warmup(
    optimizer_p2,
    num_warmup_steps   = total_steps_p2 // 10,
    num_training_steps = total_steps_p2
)

best_p2_f1   = 0
patience_cnt = 0

print(f'\n{"="*50}')
print(f'PHASE 2 — Full Fine-Tune ({EPOCHS_P2} epochs)')
print(f'Base LR: {LR_P2}  |  Head LR: {LR_P2 * LR_HEAD_MULT}')
print(f'{"="*50}\n')

for epoch in range(EPOCHS_P2):
    train_loss, train_f1 = train_one_epoch(
        model, train_loader, optimizer_p2, scheduler_p2,
        epoch=epoch, phase='phase2'
    )
    val_f1 = validate(model, val_loader)

    wandb.log({
        'phase2/epoch'      : epoch + 1,
        'phase2/train_loss' : train_loss,
        'phase2/train_f1'   : train_f1,
        'phase2/val_f1'     : val_f1,
    })

    print(f'Epoch {epoch+1}/{EPOCHS_P2}')
    print(f'  Train Loss : {train_loss:.4f}')
    print(f'  Train F1   : {train_f1:.4f}')
    print(f'  Val F1     : {val_f1:.4f}')

    if val_f1 > best_p2_f1:
        best_p2_f1   = val_f1
        patience_cnt = 0
        torch.save(model.state_dict(), '/kaggle/working/best_model_phase2.pth')
        print(f'  Saved best Phase 2 model (val F1: {best_p2_f1:.4f})')
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE_P2:
            print('  Early stopping Phase 2.')
            break

wandb.run.summary['phase2_best_val_f1'] = best_p2_f1
print(f'\nPhase 2 done. Best val F1: {best_p2_f1:.4f}')

## 🔷 Phase 6: Evaluation and Error Analysis

### 🎯 Objective

Evaluate the best-performing model on the validation set, compute detailed per-class metrics, and analyze prediction errors using a confusion matrix.

---

### ⚙️ What Happens in This Phase

* Load the **best checkpoint** from Phase 5 (fine-tuned AST model)

* Run inference on the **validation dataset**

* Generate predictions for all samples

* Compute evaluation metrics:

  * **Classification report** (precision, recall, F1-score per genre)
  * **Macro F1-score** for overall performance

* Visualize results:

  * Plot a **confusion matrix**
  * Identify correctly and incorrectly classified genres

* Log outputs:

  * Save metrics and visualizations
  * Track results using tools like **W&B**

---

### ❓ Why This Phase Is Important

* Provides **fine-grained performance insights** beyond overall accuracy
* Helps identify **class-specific weaknesses** (e.g., similar genres)
* Confusion matrix reveals **systematic misclassifications**
* Supports **model improvement decisions** (data, augmentation, architecture)
* Ensures results are **interpretable and report-ready**

---

### 💡 Key Insight

Understanding *where* the model fails is as important as knowing *how well* it performs—error analysis guides meaningful improvements.


In [ ]:
# Phase 6 — Validation evaluation and confusion matrix

from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

model.load_state_dict(
    torch.load('/kaggle/working/best_model_phase2.pth')
)
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for batch in tqdm(val_loader, desc='Evaluating'):
        input_values = batch['input_values'].to(DEVICE)
        labels       = batch['labels'].to(DEVICE)
        outputs      = model(input_values=input_values)
        preds        = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

report = classification_report(
    all_labels, all_preds,
    target_names = GENRES,
    digits       = 3,
    output_dict  = True
)
print(classification_report(
    all_labels, all_preds,
    target_names=GENRES,
    digits=3
))

for genre in GENRES:
    wandb.log({f'val_f1/{genre}': report[genre]['f1-score']})

fig, ax = plt.subplots(figsize=(10, 8))
cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=GENRES, yticklabels=GENRES, ax=ax)
ax.set_title('Confusion Matrix — Phase 2 Best Model')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
plt.tight_layout()
wandb.log({'eval/confusion_matrix': wandb.Image(fig)})
plt.show()

## 🔷 Phase 7: Inference with TTA and Submission

### 🎯 Objective

Perform robust inference on the test set using Test-Time Augmentation (TTA), generate the final `submission.csv`, and complete experiment tracking.

---

### ⚙️ What Happens in This Phase

* Implement **TTA (Test-Time Augmentation)**:

  * Generate multiple crops/variants of each test audio sample
  * Run inference on each variant
  * Average predicted probabilities for final prediction

* Run **model inference**:

  * Use the best fine-tuned model
  * Predict genre probabilities for each test sample

* Create **submission file**:

  * Construct a `submission.csv` with:

    * Sample ID
    * Predicted genre label

* Log results:

  * Plot prediction distribution across genres
  * Save outputs and metrics to **Weights & Biases (W&B)**

* Finalize experiment:

  * Close W&B run
  * Ensure all artifacts are stored

---

### ❓ Why This Phase Is Important

* TTA improves **prediction stability and robustness**
* Reduces sensitivity to input variations (e.g., crop position)
* Ensures **better generalization on unseen data**
* Produces a **competition-ready submission file**
* Maintains proper experiment tracking and reproducibility

---

### 💡 Key Insight

Averaging predictions over multiple augmented inputs (TTA) often yields better performance than relying on a single deterministic prediction.


In [ ]:
# Phase 7 — TTA prediction function

def predict_with_tta(model, audio, n_tta=N_TTA):
    all_probs = []
    for _ in range(n_tta):
        cropped = crop_random(audio)
        cropped = normalize(cropped)
        inputs  = feature_extractor(
            cropped,
            sampling_rate  = SAMPLE_RATE,
            return_tensors = 'pt'
        )
        input_values = inputs['input_values'].to(DEVICE)
        with torch.no_grad():
            outputs = model(input_values=input_values)
            probs   = torch.softmax(outputs.logits, dim=1)
            all_probs.append(probs.cpu().numpy())

    avg_probs = np.mean(all_probs, axis=0)
    return np.argmax(avg_probs, axis=1)[0]

print(f'TTA function ready — {N_TTA} crops per prediction.')

In [ ]:
# Phase 7 — Generate submission on test set

model.load_state_dict(
    torch.load('/kaggle/working/best_model_phase2.pth')
)
model.eval()

test_df = pd.read_csv(os.path.join(BASE_PATH, 'test.csv'))
print(f'Test samples: {len(test_df)}')

all_ids, all_preds = [], []

for idx in tqdm(range(len(test_df)), desc=f'Inference (TTA x{N_TTA})'):
    row   = test_df.iloc[idx]
    path  = os.path.join(BASE_PATH, row['filename'])
    audio = load_audio(path)
    pred  = predict_with_tta(model, audio, n_tta=N_TTA)
    all_ids.append(row['id'])
    all_preds.append(id2label[pred])

submission = pd.DataFrame({'id': all_ids, 'genre': all_preds})
submission.to_csv('/kaggle/working/submission.csv', index=False)

print(f'Submission saved — {len(submission)} rows')
print('\nGenre distribution:')
print(submission['genre'].value_counts())

fig2, ax2 = plt.subplots(figsize=(10, 4))
submission['genre'].value_counts().plot(kind='bar', ax=ax2,
                                        color='steelblue',
                                        edgecolor='white')
ax2.set_title('Submission Genre Distribution')
ax2.set_xlabel('Genre')
ax2.set_ylabel('Count')
plt.tight_layout()
wandb.log({'submission/genre_distribution': wandb.Image(fig2)})
plt.show()

In [ ]:
# Phase 7 — Final summary and W&B close

wandb.run.summary['final_val_f1']    = best_p2_f1
wandb.run.summary['n_tta']           = N_TTA
wandb.run.summary['expected_kaggle'] = round(best_p2_f1 + 0.02, 4)

print('=' * 50)
print('TRAINING COMPLETE')
print('=' * 50)
print(f'  Phase 1 best val F1 : {best_p1_f1:.4f}')
print(f'  Phase 2 best val F1 : {best_p2_f1:.4f}')
print(f'  TTA crops used      : {N_TTA}')
print(f'  Expected Kaggle F1  : ~{best_p2_f1 + 0.02:.4f}')
print('=' * 50)
print()
print('To improve further, try:')
print('  DURATION   = 25     (more audio context)')
print('  TRAIN_SIZE = 6000   (more diversity)')
print('  EPOCHS_P2  = 7      (more fine-tuning)')
print('  LR_P2      = 1e-5   (safer updates)')
print('  N_TTA      = 7      (better inference)')

wandb.finish()
print('\nW&B run closed.')

In [1]:
import wandb
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
wandb.login(key=user_secrets.get_secret("WANDB_API_KEY"))

api  = wandb.Api()
runs = api.runs("iitanish-indian-institute-of-technology-madras/music-genre-classification")

for run in runs:
    print(f"\nRun: {run.name} | State: {run.state}")
    print("\n=== SUMMARY ===")
    for k, v in sorted(run.summary.items()):
        if not k.startswith("_"):
            print(f"  {k}: {v}")
    print("\n=== EPOCH HISTORY ===")
    h = run.history(samples=200)
    cols = [c for c in h.columns if any(x in c for x in
            ['phase1/val','phase1/train','phase2/val','phase2/train','val_f1/'])]
    print(h[cols].dropna(how='all').to_string())

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: iitanish (iitanish-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



Run: AST_dur20_bs6_p2ep5 | State: finished

=== SUMMARY ===
  dataset/genres: 10
  dataset/noise_files: 2000
  dataset/total_songs: 1000
  dataset/train_samples: 5000
  dataset/val_samples: 150
  eval/confusion_matrix: {'_type': 'image-file', 'format': 'png', 'height': 800, 'path': 'media/images/eval/confusion_matrix_3357_498ee870f1f0c5156e22.png', 'sha256': '498ee870f1f0c5156e22e01803e0f112169b376f60e5b3a6c8ab7b50177c3972', 'size': 44905, 'width': 1000}
  expected_kaggle: 0.8588
  final_val_f1: 0.8388153314590454
  model/total_params: 86196490
  n_tta: 5
  phase1/epoch: 3
  phase1/lr: 0
  phase1/step_loss: 0.7440985441207886
  phase1/train_f1: 0.9093215465545654
  phase1/train_loss: 0.7787029458750352
  phase1/val_f1: 0.8176988363265991
  phase1_best_val_f1: 0.8240617513656616
  phase2/epoch: 5
  phase2/lr: 0
  phase2/step_loss: 0.5049634575843811
  phase2/train_f1: 0.9884806871414183
  phase2/train_loss: 0.5318114253685629
  phase2/val_f1: 0.8323764801025391
  phase2_best_val_f1: 0.